🛠️ Step-by-Step Implementation 

Step 1: Hash Both Tables
Add a row_hash column to both the cdc_df and the active records of the dim_df.
(Hint: Use F.md5(F.concat_ws("||", F.col("email"), F.col("city"))))

Step 2: Join
Do a Left join from the cdc_df to the active dim_df on customer_id.

Step 3: Filter Out the Noise
Filter out the rows where the CDC hash equals the Silver hash (e.g., Bob).

Step 4: Tag the Rest
Use an F.when().otherwise() statement. If the Silver ID is null, tag as "INSERT". If the Silver ID exists but the hashes are different, tag as "UPDATE".

In [1]:
from pyspark.sql import SparkSession
import getpass

username = getpass.getuser()

In [2]:
spark = SparkSession.builder \
.config("spark.port.ui", 0) \
.config("spark.sql.warehouse.dir", f"/user/{username}/warehouse") \
.enableHiveSupport() \
.master("yarn") \
.getOrCreate()

In [3]:
from pyspark.sql.types import *
import pyspark.sql.functions as F
from datetime import date

# 1. Existing Dimension Table (Silver layer)
dim_schema = StructType([
    StructField("customer_id", StringType(), True),
    StructField("email", StringType(), True),
    StructField("city", StringType(), True),
    StructField("effective_date", DateType(), True),
    StructField("end_date", DateType(), True),
    StructField("is_current", BooleanType(), True)
])

dim_data = [
    ('C001', 'alice@test.com', 'New York', date(2025, 1, 1), date(2099, 12, 31), True),
    ('C002', 'bob@test.com', 'Chicago', date(2025, 1, 1), date(2099, 12, 31), True),
    ('C003', 'charlie@test.com', 'Boston', date(2025, 1, 1), date(2026, 1, 15), False),
    ('C003', 'charlie@test.com', 'Miami', date(2026, 1, 16), date(2099, 12, 31), True) # Charlie moved previously
]
dim_df = spark.createDataFrame(dim_data, dim_schema)

# 2. Today's CDC Updates from Bronze
cdc_schema = StructType([
    StructField("customer_id", StringType(), True),
    StructField("email", StringType(), True),
    StructField("city", StringType(), True),
    StructField("update_date", DateType(), True) # The date the change occurred
])

cdc_data = [
    ('C001', 'alice@test.com', 'San Francisco', date(2026, 6, 2)), # UPDATE: Alice moved
    ('C004', 'david@test.com', 'Austin', date(2026, 6, 2)),        # INSERT: Brand new customer
    ('C002', 'bob@test.com', 'Chicago', date(2026, 6, 2))          # IGNORE: Bob's data is identical to Silver
]
cdc_df = spark.createDataFrame(cdc_data, cdc_schema)

display(dim_df)
display(cdc_df)

customer_id,email,city,effective_date,end_date,is_current
C001,alice@test.com,New York,2025-01-01,2099-12-31,true
C002,bob@test.com,Chicago,2025-01-01,2099-12-31,true
C003,charlie@test.com,Boston,2025-01-01,2026-01-15,false
C003,charlie@test.com,Miami,2026-01-16,2099-12-31,true


customer_id,email,city,update_date
C001,alice@test.com,San Francisco,2026-06-02
C004,david@test.com,Austin,2026-06-02
C002,bob@test.com,Chicago,2026-06-02


🔑 Why do we only hash "Non-Key / Attribute" columns?

In a dimension table, we generally have three types of columns:

    Keys: customer_id (Business Key) or a Surrogate Key.

    Metadata (Audit Columns): effective_date, end_date, is_current, update_date.

    Attributes (Business Data): email, city, phone_number.

We only want to hash the Attributes. Here is exactly why you don't include the Keys or the Metadata:

1. The Danger of Hashing Metadata (The "False Update" Trap)
Look at the cdc_df. It has an update_date. If Alice goes into her app profile and clicks "Save" without actually changing her email or city, the source database might still spit out a CDC record with a brand new update_date of today.

If you include update_date in your hash:

    Silver Hash: md5(alice@test.com + New York + 2025-01-01)

    CDC Hash: md5(alice@test.com + New York + 2026-06-02)

The hashes won't match! Your pipeline will think Alice moved, so it will expire her old record and insert a new one. Over a year, you might end up with 50 rows for Alice that all say "New York". You just wasted storage and compute on False Updates. We only care if the business data changed.

2. The Redundancy of Hashing Keys
You could include customer_id in the hash, and it wouldn't break anything, but it's redundant. We are explicitly joining the two tables on="customer_id". We already know the IDs match; the hash is strictly to answer the question: "Did the rest of the data change?"

In [4]:
# Step 1: Hash Both Tables
silver_temp_df = dim_df.filter("is_current == 'true'").withColumn("silver_row_hash", F.md5(F.concat_ws("||", F.col("email"), F.col("city"))))
cdc_temp_df = cdc_df.withColumn("cdc_row_hash", F.md5(F.concat_ws("||", F.col("email"), F.col("city"))))

In [5]:
# Step 2: Join
joined_df = cdc_temp_df.alias("source").join(silver_temp_df.alias("target"), on="customer_id", how="left")

In [6]:
joined_df.show()

+-----------+--------------+-------------+-----------+--------------------+--------------+--------+--------------+----------+----------+--------------------+
|customer_id|         email|         city|update_date|        cdc_row_hash|         email|    city|effective_date|  end_date|is_current|     silver_row_hash|
+-----------+--------------+-------------+-----------+--------------------+--------------+--------+--------------+----------+----------+--------------------+
|       C004|david@test.com|       Austin| 2026-06-02|a75d5cffa7d3613c7...|          null|    null|          null|      null|      null|                null|
|       C001|alice@test.com|San Francisco| 2026-06-02|c8cc08e07767105cc...|alice@test.com|New York|    2025-01-01|2099-12-31|      true|a80ecdebb44f8cb99...|
|       C002|  bob@test.com|      Chicago| 2026-06-02|e1e3f04258983964d...|  bob@test.com| Chicago|    2025-01-01|2099-12-31|      true|e1e3f04258983964d...|
+-----------+--------------+-------------+----------

In [7]:
# Step 3: Filter Out the Noise
updates_and_inserts_df = joined_df.filter((F.col("cdc_row_hash") != F.col("silver_row_hash")) | F.col("silver_row_hash").isNull())

Shorter, null-safe equivalent
```updates_and_inserts_df = joined_df.filter(F.col("cdc_row_hash").isNotEqualTo(F.col("silver_row_hash"))) ```

In [8]:
updates_and_inserts_df.show()

+-----------+--------------+-------------+-----------+--------------------+--------------+--------+--------------+----------+----------+--------------------+
|customer_id|         email|         city|update_date|        cdc_row_hash|         email|    city|effective_date|  end_date|is_current|     silver_row_hash|
+-----------+--------------+-------------+-----------+--------------------+--------------+--------+--------------+----------+----------+--------------------+
|       C004|david@test.com|       Austin| 2026-06-02|a75d5cffa7d3613c7...|          null|    null|          null|      null|      null|                null|
|       C001|alice@test.com|San Francisco| 2026-06-02|c8cc08e07767105cc...|alice@test.com|New York|    2025-01-01|2099-12-31|      true|a80ecdebb44f8cb99...|
+-----------+--------------+-------------+-----------+--------------------+--------------+--------+--------------+----------+----------+--------------------+



In [9]:
# Step 4: Tag the incoming CDC rows with an action column
tagged_df = updates_and_inserts_df.withColumn(
    "action",
    F.when(F.col("silver_row_hash").isNull(), "INSERT")
     .otherwise("UPDATE")
)

In [10]:
tagged_df.show()

+-----------+--------------+-------------+-----------+--------------------+--------------+--------+--------------+----------+----------+--------------------+------+
|customer_id|         email|         city|update_date|        cdc_row_hash|         email|    city|effective_date|  end_date|is_current|     silver_row_hash|action|
+-----------+--------------+-------------+-----------+--------------------+--------------+--------+--------------+----------+----------+--------------------+------+
|       C004|david@test.com|       Austin| 2026-06-02|a75d5cffa7d3613c7...|          null|    null|          null|      null|      null|                null|INSERT|
|       C001|alice@test.com|San Francisco| 2026-06-02|c8cc08e07767105cc...|alice@test.com|New York|    2025-01-01|2099-12-31|      true|a80ecdebb44f8cb99...|UPDATE|
+-----------+--------------+-------------+-----------+--------------------+--------------+--------+--------------+----------+----------+--------------------+------+



In [11]:
final_cdc_df = tagged_df.select("customer_id", "source.email", "source.city", "update_date", "cdc_row_hash", "action")

In [12]:
final_cdc_df.show()

+-----------+--------------+-------------+-----------+--------------------+------+
|customer_id|         email|         city|update_date|        cdc_row_hash|action|
+-----------+--------------+-------------+-----------+--------------------+------+
|       C004|david@test.com|       Austin| 2026-06-02|a75d5cffa7d3613c7...|INSERT|
|       C001|alice@test.com|San Francisco| 2026-06-02|c8cc08e07767105cc...|UPDATE|
+-----------+--------------+-------------+-----------+--------------------+------+



Next Steps:

In a Slowly Changing Dimension (SCD) Type 2, an "UPDATE" actually requires two operations on the target table:

    Update the old record: Set is_current = False and end_date = update_date.

    Insert the new record: Insert the new row with is_current = True and effective_date = update_date.
    
Because Delta Lake's MERGE INTO command matches on a unique key, you cannot update a row and insert a row for the exact same key in a single pass unless you use a specific structural trick: The Null Merge Key Strategy.

In [13]:
# 1. Prepare the rows that need to be INSERTED (Brand new records + New active versions of updated records)
inserts_df = final_cdc_df.filter(F.col("action").isin("INSERT", "UPDATE")) \
    .withColumn("merge_key", F.lit(None)) # A NULL merge key guarantees it won't match, forcing an INSERT

# 2. Prepare the rows that need to UPDATE existing records (Expiring the old active versions)
updates_df = final_cdc_df.filter(F.col("action") == "UPDATE") \
    .withColumn("merge_key", F.col("customer_id"))

# 3. Combine them into our final staging DataFrame
staged_updates = inserts_df.unionByName(updates_df)

In [14]:
display(staged_updates)

customer_id,email,city,update_date,cdc_row_hash,action,merge_key
C004,david@test.com,Austin,2026-06-02,a75d5cffa7d3613c7...,INSERT,null
C001,alice@test.com,San Francisco,2026-06-02,c8cc08e07767105cc...,UPDATE,null
C001,alice@test.com,San Francisco,2026-06-02,c8cc08e07767105cc...,UPDATE,C001


### Next Steps

```
from delta.tables import DeltaTable

silver_table = DeltaTable.forName(spark, "silver_customers")

silver_table.alias("target").merge(
    staged_updates.alias("source"),
    "target.customer_id = source.merge_key" # The magic happens here
).whenMatchedUpdate(
    condition="target.is_current = true",
    set={
        "is_current": F.lit(False),
        "end_date": "source.update_date"
    }
).whenNotMatchedInsert(
    values={
        "customer_id": "source.customer_id",
        "email": "source.email",
        "city": "source.city",
        "effective_date": "source.update_date", 
        "end_date": F.lit(None),
        "is_current": F.lit(True),
        "silver_row_hash": "source.cdc_row_hash"
    }
).execute()
```

Taking it one step further: Handling "Hard Deletes" (op_type == 'D')

In real-world CDC (Change Data Capture) systems like Debezium, GoldenGate, or AWS DMS, if a user gets deleted in the source database, the CDC tool sends a payload with an operation type flag (often I for Insert, U for Update, and D for Delete).

If you are asked how to handle a hard delete, you do not physically delete the row in the Silver table. Instead, you "soft delete" or "close" it to preserve history.

Here is how you would adapt the action logic:



```
final_cdc_df = joined_df.withColumn(
    "action",
    F.when(F.col("op_type") == 'D', F.lit("DELETE"))           # <-- NEW: Handle Hard Deletes
     .when(F.col("silver_customer_id").isNull(), F.lit("INSERT"))
     .when(F.col("cdc_row_hash") != F.col("silver_row_hash"), F.lit("UPDATE"))
     .otherwise(F.lit("IGNORE"))
)
```

A "DELETE" action needs to retire the current active row. So, we include it in the updates_df.

A "DELETE" action does not need a new active row inserted. So, we exclude it from the inserts_df.

```
# Inserts only take "INSERT" and "UPDATE"
inserts_df = final_cdc_df.filter(F.col("action").isin("INSERT", "UPDATE")).withColumn("merge_key", F.lit(None))

# Updates now take "UPDATE" and "DELETE"
updates_df = final_cdc_df.filter(F.col("action").isin("UPDATE", "DELETE")).withColumn("merge_key", F.col("customer_id"))

```